# HW3 Image Classification
## We strongly recommend that you run with Kaggle for this homework


# Get Data
Notes: if the links are dead, you can download the data directly from Kaggle and upload it to the workspace, or you can use the Kaggle API to directly download the data into colab.


In [ ]:
# ! wget https://www.dropbox.com/s/6l2vcvxl54b0b6w/food11.zip

In [ ]:
# ! unzip food11.zip

# Training

In [ ]:
# Import necessary packages.
import numpy as np
import pandas as pd
import torch
import os
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Subset, Dataset
from torchvision.datasets import DatasetFolder, VisionDataset

# This is for the progress bar.
from tqdm.auto import tqdm
import datetime
import pytz
import random

In [ ]:
_is_resume_ckpt = False
_resume_ckpt_name = "20251125-231511_best.ckpt"
beijing_tz = pytz.timezone('Asia/Shanghai')  # 北京时间
_exp_name = datetime.datetime.now(beijing_tz).strftime("%Y%m%d-%H%M%S")
print(f"{_exp_name}")

In [ ]:
myseed = 6666  # set a random seed for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(myseed)
torch.manual_seed(myseed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(myseed)

## **Transforms**
Torchvision provides lots of useful utilities for image preprocessing, data wrapping as well as data augmentation.

Please refer to PyTorch official website for details about different transforms.

In [ ]:
# Normally, We don't need augmentations in testing and validation.
# All we need here is to resize the PIL image and transform it into Tensor.
test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# However, it is also possible to use augmentation in the testing phase.
# You may use train_tfm to produce a variety of images and then test using ensemble methods
train_tfm = transforms.Compose([
    transforms.Resize((140, 140)),  # 先稍微放大，为后续的随机裁剪提供更多区域选择
    
    # 空间变换
    transforms.RandomCrop((128, 128)),  # 随机裁剪回目标尺寸，增加位置不变性
    transforms.RandomHorizontalFlip(p=0.5),  # 50%概率水平翻转，增加镜像不变性
    transforms.RandomVerticalFlip(p=0.2),  # 20%概率垂直翻转，适用于对称场景
    transforms.RandomRotation(degrees=10),  # 随机旋转±10度，增加旋转不变性
    
    # 颜色变换
    transforms.ColorJitter(
        brightness=0.1,  # 亮度调整幅度为10%
        contrast=0.1,    # 对比度调整幅度为10%
        saturation=0.1,  # 饱和度调整幅度为10%
        hue=0.1          # 色调调整幅度为10%（-0.1到+0.1）
    ),  # 随机调整颜色属性，增加对光照和颜色变化的鲁棒性
    
    transforms.ToTensor(),  # 将PIL图像转换为PyTorch张量，并自动归一化到[0,1]
    
    # 标准化（重要！需要根据数据集计算）
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet数据集的RGB通道均值
        std=[0.229, 0.224, 0.225]    # ImageNet数据集的RGB通道标准差
    ),  # 将张量归一化到均值为0、标准差为1的分布，加速训练收敛

    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),  # 随机擦除，增加遮挡鲁棒性
])


## **Datasets**
The data is labelled by the name, so we load images and label while calling '__getitem__'

In [ ]:
class FoodDataset(Dataset):

    def __init__(self,path,tfm=test_tfm,files = None):
        super(FoodDataset).__init__()
        self.path = path
        self.files = sorted([os.path.join(path,x) for x in os.listdir(path) if x.endswith(".jpg")])
        if files != None:
            self.files = files
        print(f"One {path} sample",self.files[0])
        self.transform = tfm
  
    def __len__(self):
        return len(self.files)
  
    def __getitem__(self,idx):
        fname = self.files[idx]
        im = Image.open(fname)
        im = self.transform(im)
        #im = self.data[idx]
        try:
            label = int(fname.split("/")[-1].split("_")[0])
        except:
            label = -1 # test has no label
        return im,label



In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        # torch.nn.MaxPool2d(kernel_size, stride, padding)
        # input 維度 [3, 128, 128]
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),  # [64, 128, 128]
            # in_channels=3：输入图像的通道数。这里是3，对应RGB三通道。
            # out_channels=64：卷积层输出的通道数，即使用64个卷积核，每个卷积核会生成一个特征图，所以输出有64个通道。
            # kernel_size=3：卷积核的大小为3x3。
            # stride=1：卷积步长为1，即每次移动1个像素。
            # padding=1：在图像的四周填充1圈0，这样对于大小为128x128的输入，填充后为130x130，然后经过3x3卷积，输出尺寸为128x128（因为(128+2*1-3)/1+1=128），所以尺寸不变。
            nn.BatchNorm2d(64),
            # num_features=64：期望输入的特征通道数，与卷积层输出通道数相同。
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [64, 64, 64]
            # 第一个参数 2：池化窗口大小 - 使用2x2的窗口进行最大池化。
            # 第二个参数 2：步长 - 池化窗口每次移动2个像素。
            # 第三个参数 0：填充 - 在图像四周填充0圈，即不填充。

            nn.Conv2d(64, 128, 3, 1, 1), # [128, 64, 64]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [128, 32, 32]

            nn.Conv2d(128, 256, 3, 1, 1), # [256, 32, 32]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [256, 16, 16]

            nn.Conv2d(256, 512, 3, 1, 1), # [512, 16, 16]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 8, 8]
            
            nn.Conv2d(512, 512, 3, 1, 1), # [512, 8, 8]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 4, 4]
        )

        self.fc = nn.Sequential(
            nn.Linear(512*4*4, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 11)
        )

    def forward(self, x):
        out = self.cnn(x)
        out = out.view(out.size()[0], -1)
        return self.fc(out)

In [ ]:
class ResNetAdapted(nn.Module):
    def __init__(self, num_classes=11):
        super(ResNetAdapted, self).__init__()
        
        # 加载预训练的ResNet18（使用ImageNet权重进行迁移学习）
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # 修改第一层卷积以适应128×128输入
        # 原始: 7×7 conv, stride=2 → 对于128输入会过度下采样
        # 改为: 3×3 conv, stride=1 → 保留更多空间信息，适配小尺寸输入
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        
        # 保留 maxpool（不使用 Identity），让特征图正常下采样
        # 特征图尺寸变化: 128 → 64 → 32 → 16 → 8 (与原始ResNet的7×7接近)
        
        # 替换最后的全连接层
        self.resnet.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        return self.resnet(x)

In [ ]:
def GetModel():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # return Classifier().to(device)
    return ResNetAdapted().to(device)

In [ ]:
batch_size = 64
_dataset_dir = "./food11"
# Construct datasets.
# The argument "loader" tells how torchvision reads the data.
train_set = FoodDataset(os.path.join(_dataset_dir,"training"), tfm=train_tfm)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
valid_set = FoodDataset(os.path.join(_dataset_dir,"validation"), tfm=test_tfm)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

In [ ]:
def PrintAndSaveLog(str):
    with open(f"./{_exp_name}_log.txt","a") as f:
        print(str, file=f)
        print(str)

In [ ]:
# "cuda" only when GPUs are available.
device = "cuda" if torch.cuda.is_available() else "cpu"

# The number of training epochs and patience.
n_epochs = 50
patience = 10 # If no improvement in 'patience' epochs, early stop

# Initialize trackers, these are not parameters and should not be changed
stale = 0
best_acc = 0

# Initialize a model, and put it on the device specified.
model = GetModel()
if _is_resume_ckpt:
    PrintAndSaveLog(f"Load {_resume_ckpt_name} !")
    model.load_state_dict(torch.load(_resume_ckpt_name))

# For the classification task, we use cross-entropy as the measurement of performance.
# label_smoothing=0.1 helps prevent overfitting and improves generalization
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Initialize optimizer, you may fine-tune some hyperparameters such as learning rate on your own.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Learning rate scheduler: reduce LR when validation accuracy plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

for epoch in range(n_epochs):

    # ---------- Training ----------
    # Make sure the model is in train mode before training.
    model.train()

    # These are used to record information in training.
    train_loss = []
    train_accs = []

    for batch in tqdm(train_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch
        #imgs = imgs.half()
        #print(imgs.shape,labels.shape)

        # Forward the data. (Make sure data and model are on the same device.)
        logits = model(imgs.to(device))

        # Calculate the cross-entropy loss.
        # We don't need to apply softmax before computing cross-entropy as it is done automatically.
        loss = criterion(logits, labels.to(device))

        # Gradients stored in the parameters in the previous step should be cleared out first.
        optimizer.zero_grad()

        # Compute the gradients for parameters.
        loss.backward()

        # Update the parameters with computed gradients.
        optimizer.step()

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # Record the loss and accuracy.
        train_loss.append(loss.item())
        train_accs.append(acc)
        
    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)

    # Print the information.
    PrintAndSaveLog(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

    # ---------- Validation ----------
    # Make sure the model is in eval mode so that some modules like dropout are disabled and work normally.
    model.eval()

    # These are used to record information in validation.
    valid_loss = []
    valid_accs = []

    # Iterate the validation set by batches.
    for batch in tqdm(valid_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch
        #imgs = imgs.half()

        # We don't need gradient in validation.
        # Using torch.no_grad() accelerates the forward process.
        with torch.no_grad():
            logits = model(imgs.to(device))

        # We can still compute the loss (but not the gradient).
        loss = criterion(logits, labels.to(device))

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # Record the loss and accuracy.
        valid_loss.append(loss.item())
        valid_accs.append(acc)
        #break

    # The average loss and accuracy for entire validation set is the average of the recorded values.
    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)

    # Step the LR scheduler based on validation accuracy
    scheduler.step(valid_acc)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch} finished. Current Learning Rate: {current_lr}")

    # Print the information.
    PrintAndSaveLog(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")

    # update logs
    if valid_acc > best_acc:
        PrintAndSaveLog(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f} -> best")
    else:
        PrintAndSaveLog(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")

    # save models
    if valid_acc > best_acc:
        PrintAndSaveLog(f"Best model found at epoch {epoch}, saving model")
        torch.save(model.state_dict(), f"{_exp_name}_best.ckpt") # only save best to prevent output memory exceed error
        best_acc = valid_acc
        stale = 0
    else:
        stale += 1
        if stale > patience:
            PrintAndSaveLog(f"No improvment {patience} consecutive epochs, early stopping")
            break

In [ ]:
test_set = FoodDataset(os.path.join(_dataset_dir,"test"), tfm=test_tfm)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

# Testing and generate prediction CSV

In [ ]:
model_best = GetModel()
model_best.load_state_dict(torch.load(f"{_exp_name}_best.ckpt"))
model_best.eval()
prediction = []
with torch.no_grad():
    for data,_ in test_loader:
        test_pred = model_best(data.to(device))
        test_label = np.argmax(test_pred.cpu().data.numpy(), axis=1)
        prediction += test_label.squeeze().tolist()

In [ ]:
#create test csv
def pad4(i):
    return "0"*(4-len(str(i)))+str(i)
df = pd.DataFrame()
df["Id"] = [pad4(i) for i in range(1,len(test_set)+1)]
df["Category"] = prediction
df.to_csv("submission.csv",index = False)

# Q1. Augmentation Implementation
## Implement augmentation by finishing train_tfm in the code with image size of your choice. 
## Directly copy the following block and paste it on GradeScope after you finish the code
### Your train_tfm must be capable of producing 5+ different results when given an identical image multiple times.
### Your  train_tfm in the report can be different from train_tfm in your training code.


In [ ]:
train_tfm = transforms.Compose([
    # Resize the image into a fixed shape (height = width = 128)
    transforms.Resize((128, 128)),
    # You need to add some transforms here.
    transforms.ToTensor(),
])

# Q2. Residual Implementation
![](https://i.imgur.com/GYsq1Ap.png)
## Directly copy the following block and paste it on GradeScope after you finish the code


In [ ]:
from torch import nn
class Residual_Network(nn.Module):
    def __init__(self):
        super(Residual_Network, self).__init__()
        
        self.cnn_layer1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
        )

        self.cnn_layer2 = nn.Sequential(
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
        )

        self.cnn_layer3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
        )

        self.cnn_layer4 = nn.Sequential(
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
        )
        self.cnn_layer5 = nn.Sequential(
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.BatchNorm2d(256),
        )
        self.cnn_layer6 = nn.Sequential(
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(256* 32* 32, 256),
            nn.ReLU(),
            nn.Linear(256, 11)
        )
        self.relu = nn.ReLU()

        # ========== 新增：3 个 1×1 卷积用于跳跃连接 ==========                                                                                           
        self.downsample1 = nn.Conv2d(64, 64, 1)          # Block1: 3→64 通道匹配                                                                           
        self.downsample2 = nn.Conv2d(128, 128, 1, 2)     # Block2: 通道+空间匹配                                                                           
        self.downsample3 = nn.Conv2d(256, 256, 1, 2)    # Block3: 通道+空间匹配  

    def res_block(self, block, downsample, x):
        identity = downsample(x)
        x_out = block(x) + identity 
        return x_out
    
    def forward_single_block(self, block_1, block_2, downsample, x):
        x_out = block_1(x)    
        x_out = self.relu(x_out)
        x_out = self.res_block(block_2, downsample, x_out)
        x_out = self.relu(x_out)
        return x_out


    def forward(self, x):
        # input (x): [batch_size, 3, 128, 128]
        # output: [batch_size, 11]

        # Extract features by convolutional layers.

        # ===== Block 1 =====
        x1 = self.forward_single_block(self.cnn_layer1, self.cnn_layer2, self.downsample1, x)
        x2 = self.forward_single_block(self.cnn_layer3, self.cnn_layer4, self.downsample2, x1)
        x3 = self.forward_single_block(self.cnn_layer5, self.cnn_layer6, self.downsample3, x2)
        # The extracted feature map must be flatten before going to fully-connected layers.
        xout = x3.flatten(1)

        # The features are transformed by fully-connected layers to obtain the final logits.
        xout = self.fc_layer(xout)
        return xout